# Análisis de un grupo de Telegram

En esta libreta analizo el historial de un grupo público de Telegram en ruso, dedicado a
la compraventa e intercambio de cuentas de un videojuego, entre agosto de 2025 y
septiembre de 2026. Los datos crudos no se suben al repositorio, así que la libreta se
entrega ya ejecutada. Para proteger a los participantes, los nombres se reemplazaron por
superhéroes y solo se muestran resultados agregados, nunca mensajes completos.

## Índice

Las secciones 0 a 2 preparan los datos y las siguientes responden las preguntas de la tarea.

- Sección 0: Configuración.
- Sección 1: Lectura del HTML exportado.
- Sección 2: Anonimización y formato tidy.
- Sección 3: Pregunta 1, ¿qué usuario envía más mensajes?
- Sección 4: Pregunta 2, ¿cuántas palabras escribe en promedio cada usuario por mensaje de texto?
- Sección 5: Pregunta 3, ¿quién envía más palabras, emojis y stickers?
- Sección 6: Pregunta 4, ¿qué días y a qué horas se envían más mensajes?
- Sección 7: Pregunta 5, ¿qué palabras son las más usadas, en general y por usuario?
- Sección 8: Pregunta 6, ¿cuáles son los adjetivos más usados?
- Sección 9: Preguntas adicionales, que responden a "¿qué otras preguntas interesantes puedes responder?"
- Sección 10: Conclusiones.

## 0. Configuración

Cargo las librerías y las rutas del proyecto, que están definidas en `src/config.py`.

In [ ]:
import re
import sys
from collections import Counter
from pathlib import Path

import emoji
import matplotlib.pyplot as plt
import nltk
import pandas as pd
import seaborn as sns
import spacy
from lxml import html
from nltk.corpus import stopwords

# Para poder importar src/ desde la carpeta notebooks/
PROJ_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJ_ROOT))

from src.config import CHAT_EXPORT_DIR, FIGURES_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR

pd.set_option("display.max_columns", 30)

# Estilo de las gráficas
sns.set_theme(style="whitegrid")
AZUL = "#2a78d6"  # color principal
GRIS = "#898781"  # "Otros" y líneas de referencia

## 1. Lectura del HTML exportado

Telegram exportó el chat en 35 archivos HTML (`messages.html`, `messages2.html`, ...,
`messages35.html`). Los leo en orden numérico para conservar la secuencia de los mensajes.

In [ ]:
def numero_archivo(ruta):
    # messages.html es el 1, messages2.html el 2, etc.
    return int(re.search(r"messages(\d*)\.html", ruta.name).group(1) or 1)


archivos = sorted(CHAT_EXPORT_DIR.glob("messages*.html"), key=numero_archivo)
print(f"{len(archivos)} archivos: {archivos[0].name} ... {archivos[-1].name}")

Cada mensaje de usuario es un bloque `div` con las clases `message default`; los avisos
del sistema, como los separadores de fecha, se ignoran. La multimedia no se descargó, así
que de cada foto, video o sticker solo conozco su tipo y, en los stickers, el emoji
asociado. En audios y archivos adjuntos el título es el nombre de la canción o del
archivo, por eso el tipo se deduce de la clase del bloque y ese título no se guarda.
Cuando alguien menciona a otra persona que no tiene `@usuario`, Telegram escribe su nombre
real en el texto; esas menciones se reemplazan desde aquí por `@usuario_anon`.

In [ ]:
# Tipos que Telegram escribe como título del bloque multimedia
TITULOS_MEDIA = {"Photo", "Sticker", "Video file", "Animation", "Voice message", "Video message"}


def hijo(elemento, clases):
    # Primer hijo directo que tiene todas las clases indicadas
    buscadas = set(clases.split())
    for h in elemento:
        if buscadas <= set((h.get("class") or "").split()):
            return h
    return None


def extraer_texto(div_texto):
    # Los saltos de línea vienen como <br>
    for br in div_texto.iter("br"):
        br.tail = "\n" + (br.tail or "")
    # Menciones sin @usuario: el enlace muestra el nombre real de la persona
    for a in div_texto.xpath('.//a[contains(@onclick, "ShowMentionName")]'):
        a.text = "@usuario_anon"
    return div_texto.text_content().strip()


def tipo_de_media(media):
    if media.find_class("media_poll"):
        return "Poll"
    titulo = media.find_class("title")[0].text_content().strip()
    if titulo in TITULOS_MEDIA:
        return titulo
    if media.find_class("media_audio_file"):
        return "Audio file"
    if media.find_class("media_file"):
        return "File"
    return "Other"


def contar_reacciones(body):
    total = 0
    for reaccion in body.find_class("reaction"):
        conteo = reaccion.find_class("count")
        if conteo:
            total += int(conteo[0].text_content())
        else:
            # Con pocas reacciones se muestran avatares en vez de un número
            total += len(reaccion.find_class("userpic"))
    return total


def leer_mensaje(msg):
    body = hijo(msg, "body")
    # En los reenviados, el texto y la multimedia están dentro de otro bloque
    reenviado = hijo(body, "forwarded body")
    contenido = reenviado if reenviado is not None else body

    nombre = hijo(body, "from_name")
    texto = hijo(contenido, "text")
    media = hijo(contenido, "media_wrap")
    respuesta = hijo(body, "reply_to")

    tipo_media = tipo_de_media(media) if media is not None else None
    sticker_emoji = None
    if tipo_media == "Sticker":
        # El estado del sticker se ve así: "<emoji>, <tamaño>"
        sticker_emoji = media.find_class("status")[0].text_content().split(",")[0].strip()
    responde_a_id = None
    if respuesta is not None:
        # El enlace es "#go_to_message33351" o "messages2.html#go_to_message33351"
        href = respuesta.find(".//a").get("href")
        responde_a_id = int(href.split("go_to_message")[1])

    return {
        "id_mensaje": int(msg.get("id").replace("message", "")),
        "fecha": hijo(body, "date").get("title"),
        "nombre": nombre.text_content().strip() if nombre is not None else None,
        "es_continuacion": "joined" in msg.get("class").split(),
        "reenviado": reenviado is not None,
        "es_respuesta": respuesta is not None,
        "responde_a_id": responde_a_id,
        "texto": extraer_texto(texto) if texto is not None else None,
        "tipo_media": tipo_media,
        "sticker_emoji": sticker_emoji,
        "n_reacciones": contar_reacciones(body),
    }

Los mensajes seguidos de una misma persona no repiten el nombre, así que lo relleno con
el último remitente visto. Algunos nombres aparecen como "Nombre via @bot"; me quedo con
el nombre y marco esos mensajes en la columna `via_bot`. Las horas vienen en UTC-07:00, la
zona horaria de la computadora con la que se exportó el chat; en la tabla tidy se pasan a
hora de Moscú.

In [ ]:
filas = []
for ruta in archivos:
    raiz = html.parse(str(ruta)).getroot()
    for msg in raiz.find_class("message"):
        if "default" in msg.get("class").split():
            fila = leer_mensaje(msg)
            fila["archivo"] = numero_archivo(ruta)
            filas.append(fila)

df_crudo = pd.DataFrame(filas)
df_crudo["timestamp"] = pd.to_datetime(df_crudo["fecha"], format="%d.%m.%Y %H:%M:%S UTC%z")
df_crudo["remitente"] = df_crudo["nombre"].ffill()
df_crudo["via_bot"] = df_crudo["remitente"].str.contains(" via @", regex=False)
df_crudo["remitente"] = df_crudo["remitente"].str.replace(r" via @\w+$", "", regex=True)
df_crudo["responde_a_id"] = df_crudo["responde_a_id"].astype("Int64")

df_crudo = df_crudo[[
    "archivo", "id_mensaje", "timestamp", "remitente", "via_bot", "es_continuacion",
    "reenviado", "es_respuesta", "responde_a_id", "texto", "tipo_media",
    "sticker_emoji", "n_reacciones",
]]
print(f"{len(df_crudo):,} mensajes leídos")

### Verificación de la lectura

La columna `remitente` todavía tiene los nombres reales, así que no la muestro; se
anonimiza en la sección 2. Por ahora reviso solo conteos generales.

In [ ]:
resumen = pd.Series({
    "Mensajes": len(df_crudo),
    "Remitentes distintos": df_crudo["remitente"].nunique(),
    "Mensajes de 'Deleted Account'": (df_crudo["remitente"] == "Deleted Account").sum(),
    "Con texto": df_crudo["texto"].notna().sum(),
    "Sin texto": df_crudo["texto"].isna().sum(),
    "Con multimedia": df_crudo["tipo_media"].notna().sum(),
    "Reenviados": df_crudo["reenviado"].sum(),
    "Respuestas": df_crudo["es_respuesta"].sum(),
    "Enviados con bot": df_crudo["via_bot"].sum(),
    "Con reacciones": (df_crudo["n_reacciones"] > 0).sum(),
})
print("Del", df_crudo["timestamp"].min(), "al", df_crudo["timestamp"].max())
resumen.to_frame("conteo")

In [ ]:
df_crudo["tipo_media"].value_counts(dropna=False).to_frame("mensajes")

In [ ]:
# Vista previa sin nombres ni textos
df_crudo.drop(columns=["remitente", "texto"]).head()

Se leyeron 34,868 mensajes de 1,322 remitentes. "Deleted Account" junta a todas las
cuentas eliminadas (3,073 mensajes), así que después la excluiré de los rankings por
usuario. Casi un 38 % de los mensajes no tiene texto y la gran mayoría de la multimedia
son fotos, lo que cuadra con un grupo donde se muestran capturas de cuentas en venta.

### Guardado intermedio

Guardo la tabla en `data/interim/` para no tener que volver a leer el HTML. Esa carpeta no se sube a GitHub porque todavía tiene los nombres reales.

In [ ]:
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
df_crudo.to_pickle(INTERIM_DATA_DIR / "mensajes_crudos.pkl")

## 2. Anonimización y formato tidy

Antes de cualquier análisis quito todo lo que pueda identificar a una persona: nombres,
menciones, correos, enlaces, tarjetas y teléfonos. Después armo la tabla tidy con la que se responden todas las preguntas.

### Nombres de superhéroes

Cada remitente recibe un nombre de superhéroe según su actividad: el más activo es Batman,
el segundo Superman, y así sucesivamente. Como hay más de 1,300 usuarios y solo 50 nombres,
a partir del usuario 51 se repiten con un sufijo (`Batman_2`, `Superman_2`, ...). "Deleted
Account" no es una persona sino todas las cuentas eliminadas juntas, así que la llamo
"Cuenta eliminada" y no le asigno superhéroe.

In [ ]:
HEROES = [
    "Batman", "Superman", "Flash", "Hulk", "Storm", "Thor", "Robin", "Spiderman",
    "Wolverine", "Aquaman", "Cyclops", "Rogue", "Falcon", "Groot", "Vision",
    "Hawkeye", "Blade", "Raven", "Cyborg", "Gambit", "Nightwing", "Batgirl",
    "Supergirl", "Daredevil", "Elektra", "Punisher", "Shazam", "Atom", "Starfire",
    "Beast", "Iceman", "Colossus", "Nova", "Quicksilver", "Wasp", "Hercules",
    "Magik", "Havok", "Bishop", "Cable", "Domino", "Deadpool", "Sentry",
    "Hellboy", "Zatanna", "Huntress", "Stargirl", "Mera", "Firestorm", "Rocket",
]

# Remitentes ordenados por número de mensajes (en empate, el que escribió primero)
actividad = (
    df_crudo[df_crudo["remitente"] != "Deleted Account"]
    .groupby("remitente")
    .agg(mensajes=("id_mensaje", "size"), primer_mensaje=("id_mensaje", "min"))
    .sort_values(["mensajes", "primer_mensaje"], ascending=[False, True])
)

# La tabla de equivalencias solo existe en memoria; no se guarda en ningún archivo
equivalencias = {}
for i, remitente in enumerate(actividad.index):
    vuelta = i // len(HEROES) + 1
    heroe = HEROES[i % len(HEROES)]
    equivalencias[remitente] = heroe if vuelta == 1 else f"{heroe}_{vuelta}"
equivalencias["Deleted Account"] = "Cuenta eliminada"

print(f"{len(actividad):,} usuarios anonimizados; el sufijo más alto es _{vuelta}")
print("Los 15 más activos:", ", ".join(list(equivalencias.values())[:15]))

### Limpieza del texto

En el texto cambio cada mención `@usuario` por `@usuario_anon` y borro correos, enlaces
(incluidos los de `t.me`, que suelen llevar al perfil de alguien, aunque se escriban con
letras cirílicas parecidas, y los `tg://`), números de tarjeta y
teléfonos. Como tarjeta cuento 4 grupos de 4 dígitos separados por espacios o guiones, y
como teléfono el formato ruso (`+7` u `8` seguido de 10 dígitos), los números que empiezan
con `+` y cualquier número de 10 dígitos o más. Los números cortos, como precios y
cantidades, se conservan.

In [ ]:
PATRON_CORREO = r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"
PATRON_ENLACE = (
    r"(?:https?://|www\.|tg://)\S+"
    r"|\b(?:[tт]|telegram)\.[mм][eе]/\S*"  # también "t.mе" con letras cirílicas
    r"|\b[\w-]+\.(?:ru|com|net|org|me|su|gg|io)\b(?:/\S*)?"
)
PATRON_MENCION = r"@\w+"
PATRON_TARJETA = r"\b\d{4}(?:[ -]\d{4}){3}\b"
PATRON_TELEFONO = (
    r"(?:\+7|\b8)[\s(-]*\d{3}[\s)-]*\d{3}[\s-]*\d{2}[\s-]*\d{2}\b"
    r"|\+\d[\d\s()-]{8,}\d"
    r"|\b\d{10,}\b"
)


def limpiar_texto(texto):
    # El correo va antes que los enlaces y las menciones para no dejar la parte del usuario
    texto = texto.str.replace(PATRON_CORREO, " ", regex=True)
    texto = texto.str.replace(PATRON_ENLACE, " ", regex=True, flags=re.IGNORECASE)
    texto = texto.str.replace(PATRON_MENCION, "@usuario_anon", regex=True)
    texto = texto.str.replace(PATRON_TARJETA, " ", regex=True)
    texto = texto.str.replace(PATRON_TELEFONO, " ", regex=True)
    texto = texto.str.replace(r"[ \t]+", " ", regex=True).str.strip()
    # Si el mensaje solo tenía un enlace, queda vacío y lo trato como sin texto
    return texto.mask(texto == "")


texto_limpio = limpiar_texto(df_crudo["texto"])

# Comprobación: no debe quedar ningún dato de contacto ni mención con nombre
revision = pd.Series({
    "Correos": texto_limpio.str.contains(PATRON_CORREO).sum(),
    "Enlaces": texto_limpio.str.contains(PATRON_ENLACE, flags=re.IGNORECASE).sum(),
    "Tarjetas": texto_limpio.str.contains(PATRON_TARJETA).sum(),
    "Teléfonos": texto_limpio.str.contains(PATRON_TELEFONO).sum(),
    "Menciones sin anonimizar": texto_limpio.str.contains(r"@(?!usuario_anon\b)\w+").sum(),
    "Mensajes con menciones anonimizadas": texto_limpio.str.contains("@usuario_anon", regex=False).sum(),
    "Mensajes que quedaron sin texto": (df_crudo["texto"].notna() & texto_limpio.isna()).sum(),
})
revision.to_frame("mensajes")

### Construcción de la tabla tidy

Además del texto limpio, calculo algunas columnas que se usan en las preguntas. La fecha y
hora se pasa de UTC-07:00 a hora de Moscú, que es la referencia más razonable para un grupo
en ruso; en la sección 6 explico por qué esto es un supuesto. El tipo de
contenido se toma de la multimedia; si no hay multimedia pero sí texto, el mensaje es de
texto. Cuento como palabra cualquier secuencia de letras latinas o cirílicas, sin contar números, emojis ni menciones. Por último, marco como
repetido todo texto que aparece idéntico más de 3 veces en el grupo, que casi siempre son
anuncios pegados una y otra vez.

In [ ]:
TIPOS = {
    "Photo": "foto", "Sticker": "sticker", "Animation": "gif",
    "Video file": "video", "Video message": "video", "Voice message": "voz",
}
PATRON_PALABRA = r"[a-zа-яё]+(?:-[a-zа-яё]+)*"


def tipo_contenido(tipo_media, texto):
    if pd.notna(tipo_media):
        return TIPOS.get(tipo_media, "otro")  # audios, archivos y encuestas son "otro"
    if pd.notna(texto):
        return "texto"
    return "otro"  # mensajes que solo tenían un enlace


df = pd.DataFrame({
    "id_mensaje": df_crudo["id_mensaje"],
    "timestamp": df_crudo["timestamp"].dt.tz_convert("Europe/Moscow"),
    "usuario": df_crudo["remitente"].map(equivalencias),
    "texto": texto_limpio,
    "tipo_contenido": [tipo_contenido(m, t) for m, t in zip(df_crudo["tipo_media"], texto_limpio)],
    "reenviado": df_crudo["reenviado"],
    "es_respuesta": df_crudo["es_respuesta"],
    "responde_a_id": df_crudo["responde_a_id"],
    "sticker_emoji": df_crudo["sticker_emoji"],
    "n_reacciones": df_crudo["n_reacciones"],
    "via_bot": df_crudo["via_bot"],
})

veces_texto = df["texto"].map(df["texto"].value_counts())
df["es_repetido"] = veces_texto > 3

sin_menciones = df["texto"].str.replace("@usuario_anon", " ", regex=False)
df["n_palabras"] = sin_menciones.str.count(PATRON_PALABRA, flags=re.IGNORECASE).fillna(0).astype(int)
df["n_emojis"] = df["texto"].fillna("").map(emoji.emoji_count)

print(f"{len(df):,} filas y {df.shape[1]} columnas; usuarios sin nombre: {df['usuario'].isna().sum()}")
# Vista previa sin texto ni usuario: junto con la hora exacta podrían identificar a alguien
df.drop(columns=["texto", "usuario"]).head()

In [ ]:
tipos = df["tipo_contenido"].value_counts().to_frame("mensajes")
tipos["porcentaje"] = (100 * tipos["mensajes"] / len(df)).round(1)
tipos

In [ ]:
pd.Series({
    "Mensajes con texto": df["texto"].notna().sum(),
    "Mensajes repetidos (texto idéntico > 3 veces)": df["es_repetido"].sum(),
    "Textos distintos que se repiten": df.loc[df["es_repetido"], "texto"].nunique(),
    "Palabras en total": df["n_palabras"].sum(),
    "Palabras en mensajes no repetidos": df.loc[~df["es_repetido"], "n_palabras"].sum(),
    "Emojis en total": df["n_emojis"].sum(),
    "Mensajes de 'Cuenta eliminada'": (df["usuario"] == "Cuenta eliminada").sum(),
    "Fotos con descripción": ((df["tipo_contenido"] == "foto") & df["texto"].notna()).sum(),
}).to_frame("conteo")

El 80 % de los mensajes son fotos (muchas con un texto como descripción) y solo el 17.5 %
son mensajes únicamente de texto; stickers, GIFs y audios casi no se usan. De los 21,730
mensajes con texto, 8,404 son repeticiones de apenas 592 textos distintos, y quitarlos deja
160 mil de las 256 mil palabras. Por eso en el análisis de palabras se excluyen los
repetidos, porque si no los anuncios dominarían los resultados.

### ¿Por qué la tabla es tidy?

La tabla cumple las tres reglas de los datos tidy: cada fila es un mensaje, cada columna es
una sola variable y cada celda guarda un único valor. Por ejemplo, el tipo de contenido
está en una sola columna en lugar de tener una columna por tipo, y la fecha y la hora
están juntas en `timestamp`, de donde se pueden sacar el día o la hora cuando se necesiten.
La descripción de cada columna está en el
[diccionario de datos](../references/diccionario_de_datos.md).

El flujo completo es: los HTML de `data/raw/` se leen a una tabla cruda que se guarda en
`data/interim/`; después se anonimiza y limpia, y la tabla tidy resultante se guarda en
`data/processed/`, de donde parte todo el análisis.

### Guardado de la tabla limpia

La tabla tidy se guarda en `data/processed/`. Aunque ya no tiene nombres reales, sí tiene
los textos de los mensajes, así que esa carpeta tampoco se sube a GitHub. A partir de aquí
todo el análisis usa solo `df`.

In [ ]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
df.to_pickle(PROCESSED_DATA_DIR / "mensajes_tidy.pkl")
df.dtypes.to_frame("tipo")

## 3. Pregunta 1: ¿Qué usuario envía más mensajes?

Cuento todos los mensajes de cada usuario, sean fotos, texto u otro contenido. "Cuenta
eliminada" queda fuera del ranking porque junta a muchas personas distintas; solo la muestro
como referencia. Los 15 usuarios más activos aparecen por separado y los demás se agrupan
como "Otros"; esta misma agrupación se usa en las preguntas 2 y 3.

In [ ]:
# Tabla sin "Cuenta eliminada" que se usa en todas las preguntas por usuario
usuarios = df[df["usuario"] != "Cuenta eliminada"].copy()
mensajes_por_usuario = usuarios["usuario"].value_counts()
top15 = mensajes_por_usuario.head(15).index.tolist()
orden = top15 + ["Otros"]
usuarios["grupo"] = usuarios["usuario"].where(usuarios["usuario"].isin(top15), "Otros")


def barras(ax, serie, titulo, etiqueta_x, formato="{:,.0f}"):
    # Barras horizontales ordenadas de arriba hacia abajo, con "Otros" en gris
    colores = [GRIS if nombre == "Otros" else AZUL for nombre in serie.index]
    ax.barh(serie.index, serie.values, color=colores)
    ax.invert_yaxis()
    # Fondo blanco en los números para que se lean aunque una línea pase por detrás
    ax.bar_label(ax.containers[0], fmt=formato, padding=3, fontsize=9,
                 bbox=dict(facecolor="white", edgecolor="none", pad=1))
    ax.set_xlim(0, serie.max() * 1.15)
    ax.set_title(titulo)
    ax.set_xlabel(etiqueta_x)
    ax.grid(axis="y", visible=False)


# Los 15 más activos, "Otros" y "Cuenta eliminada" como referencia
grupo = df["usuario"].where(df["usuario"].isin(top15 + ["Cuenta eliminada"]), "Otros")
tabla_p1 = pd.DataFrame({
    "mensajes": df.groupby(grupo).size(),
    "% del total": 100 * df.groupby(grupo).size() / len(df),
    "% fotos": 100 * (df["tipo_contenido"] == "foto").groupby(grupo).mean(),
    "% repetidos": 100 * df["es_repetido"].groupby(grupo).mean(),
})
tabla_p1.loc[orden + ["Cuenta eliminada"]].round(1)

La gráfica de la izquierda muestra a los 15 más activos. La de la derecha es un boxplot con
todos los usuarios; uso escala logarítmica porque unos pocos usuarios tienen cientos de
mensajes y la mayoría solo unos cuantos, y en escala normal la caja quedaría aplastada
contra el cero.

In [ ]:
print(f"{len(mensajes_por_usuario):,} usuarios; mediana de "
      f"{mensajes_por_usuario.median():.0f} mensajes por usuario")
print(f"Usuarios con un solo mensaje: {(mensajes_por_usuario == 1).sum()}")
print(f"Los 15 más activos enviaron el "
      f"{100 * mensajes_por_usuario.head(15).sum() / len(df):.1f} % de todos los mensajes")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [3, 2]})
barras(ax1, mensajes_por_usuario.head(15), "Los 15 usuarios con más mensajes", "Mensajes")

sns.boxplot(x=mensajes_por_usuario.values, ax=ax2, color=AZUL, log_scale=True, width=0.4)
ax2.set_title(f"Mensajes por usuario ({len(mensajes_por_usuario):,} usuarios)")
ax2.set_xlabel("Mensajes (escala logarítmica)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "p1_mensajes_por_usuario.png", dpi=120, bbox_inches="tight")
plt.show()

### Interpretación

Batman es el usuario que más mensajes envía (1,363, el 3.9 % del total), seguido de
Superman (990) y Flash (755), y casi todos sus mensajes son fotos, probablemente capturas de cuentas. "Cuenta
eliminada" suma todavía más (3,073), pero junta a muchas personas y por eso no cuenta. La
actividad está muy concentrada: la mitad de los 1,321 usuarios envió 5 mensajes o menos y
309 escribieron una sola vez, mientras que los 15 más activos suman casi una cuarta parte
de todo el chat (24.8 %). Además, algunos de los más activos, como Hulk (99.9 %) y Rogue
(83.7 %), publican casi solo anuncios repetidos.

## 4. Pregunta 2: ¿Cuántas palabras escribe en promedio cada usuario por mensaje de texto?

Aquí solo cuento los mensajes con al menos una palabra, según la columna `n_palabras` de la
sección 2. Incluyo las descripciones de las fotos, porque en este grupo la mayor parte del
texto acompaña a una captura; los mensajes con solo emojis o números quedan fuera. El
promedio principal incluye los anuncios repetidos, y en la tabla agrego el promedio sin
ellos para comparar.

In [ ]:
con_palabras = usuarios[usuarios["n_palabras"] > 0]
no_repetidos = con_palabras[~con_palabras["es_repetido"]]
solo_texto = con_palabras[con_palabras["tipo_contenido"] == "texto"]

print(f"Promedio del grupo: {con_palabras['n_palabras'].mean():.1f} palabras "
      f"({len(con_palabras):,} mensajes)")
print(f"Sin repetidos: {no_repetidos['n_palabras'].mean():.1f} palabras "
      f"({len(no_repetidos):,} mensajes)")
print(f"Solo mensajes únicamente de texto: {solo_texto['n_palabras'].mean():.1f} palabras "
      f"({len(solo_texto):,} mensajes)")

tabla_p2 = pd.DataFrame({
    "mensajes con texto": con_palabras.groupby("grupo").size(),
    "promedio": con_palabras.groupby("grupo")["n_palabras"].mean(),
    "mediana": con_palabras.groupby("grupo")["n_palabras"].median(),
    "promedio sin repetidos": no_repetidos.groupby("grupo")["n_palabras"].mean(),
}).loc[orden]
tabla_p2.round(1)

En el boxplot de la derecha entran todos los usuarios con al menos 5 mensajes de texto;
con menos, el promedio depende de uno o dos mensajes y no dice mucho de cómo escribe esa
persona. Uso otra vez escala logarítmica porque unos pocos usuarios escriben mensajes muy
largos y aplastarían la caja.

In [ ]:
promedio_usuario = con_palabras.groupby("usuario")["n_palabras"].mean()
mensajes_texto = con_palabras.groupby("usuario").size()
promedio_5 = promedio_usuario[mensajes_texto >= 5]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [3, 2]})
barras(ax1, tabla_p2["promedio"], "Palabras promedio por mensaje de texto",
       "Palabras por mensaje", formato="{:.1f}")
ax1.axvline(con_palabras["n_palabras"].mean(), color=GRIS, linestyle="--", zorder=0,
            label=f"Promedio del grupo ({con_palabras['n_palabras'].mean():.1f})")
ax1.legend(loc="lower right")

sns.boxplot(x=promedio_5.values, ax=ax2, color=AZUL, log_scale=True, width=0.4)
ax2.set_title(f"Promedio por usuario ({len(promedio_5)} usuarios con 5 o más mensajes)")
ax2.set_xlabel("Palabras por mensaje (escala logarítmica)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "p2_palabras_por_mensaje.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Mediana del promedio por usuario: {promedio_5.median():.1f} palabras")

### Interpretación

Un mensaje con texto tiene en promedio 11.6 palabras, y el usuario típico (la mediana del
boxplot) escribe unas 10 por mensaje: son mensajes cortos, como descripciones de cuentas o
precios. Entre los 15 más activos, Robin (22.9) y Hulk (21.0) escriben los mensajes más
largos, y Flash (2.9) y Wolverine (3.2) los más cortos. Los repetidos cambian algunos
casos: Hulk solo publica anuncios repetidos y Batman sube de 11.4 a 22.3 palabras al
quitarlos. Si solo se cuentan los mensajes únicamente de texto, sin las descripciones de
fotos, el promedio del grupo sube de 11.6 a 13.0 palabras, así que las descripciones son un
poco más cortas que los mensajes normales.

## 5. Pregunta 3: ¿Quién envía más palabras, emojis y stickers?

Para cada usuario sumo todas sus palabras, emojis y stickers. Los rankings incluyen a todos
los usuarios (menos "Cuenta eliminada"), porque quien más escribe no siempre es quien más
mensajes manda. Como los anuncios repetidos inflan mucho el conteo de palabras, muestro ese
ranking con y sin ellos. Un sticker es un mensaje cuyo `tipo_contenido` es "sticker".

In [ ]:
usuarios["palabras_sin_repetir"] = usuarios["n_palabras"].where(~usuarios["es_repetido"], 0)
usuarios["es_sticker"] = usuarios["tipo_contenido"] == "sticker"

columnas = {
    "n_palabras": "palabras",
    "palabras_sin_repetir": "palabras sin repetidos",
    "n_emojis": "emojis",
    "es_sticker": "stickers",
}
totales = usuarios.groupby("usuario")[list(columnas)].sum().rename(columns=columnas)

# Los 15 más activos y "Otros"
tabla_p3 = usuarios.groupby("grupo")[list(columnas)].sum().rename(columns=columnas)
tabla_p3.loc[orden]

Las gráficas muestran los 10 primeros de todo el grupo en palabras y emojis. Como los
stickers son muy pocos, en ese panel solo aparecen los usuarios con al menos 3 stickers.

In [ ]:
stickers_3 = totales["stickers"][totales["stickers"] >= 3].sort_values(ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
barras(axes[0, 0], totales["palabras"].nlargest(10), "Palabras (con repetidos)", "Palabras")
barras(axes[0, 1], totales["palabras sin repetidos"].nlargest(10), "Palabras sin repetidos", "Palabras")
barras(axes[1, 0], totales["emojis"].nlargest(10), "Emojis", "Emojis")
barras(axes[1, 1], stickers_3, "Stickers (usuarios con 3 o más)", "Stickers")

fig.suptitle("Usuarios que más palabras, emojis y stickers envían", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "p3_palabras_emojis_stickers.png", dpi=120, bbox_inches="tight")
plt.show()

Cada sticker de Telegram lleva un emoji asociado, que el HTML exportado sí conserva. La
tabla muestra los más comunes.

In [ ]:
# Emojis asociados a los stickers (todo el grupo)
print(f"Stickers en total: {(df['tipo_contenido'] == 'sticker').sum()} "
      f"({100 * (df['tipo_contenido'] == 'sticker').mean():.1f} % de los mensajes)")
print(f"Usuarios que enviaron al menos un sticker: {(totales['stickers'] > 0).sum()}")
df["sticker_emoji"].value_counts().head(5).rename_axis("emoji").to_frame("stickers")



### Interpretación

Con los anuncios incluidos, Hulk es quien más palabras envía (15,435), pero todas vienen de
un mismo anuncio repetido; sin repetidos, el primero es Hawkeye (4,221), seguido de Thor
(3,436). En emojis gana Stargirl (923), que ni siquiera está entre los 15 más activos, y
Hulk vuelve a quedar segundo, probablemente porque su anuncio lleva emojis. Los stickers son solo el
0.3 % de los mensajes (114) y los envían 49 usuarios; Cable_2 manda 32, casi un tercio del
total, y el emoji más asociado a los stickers es 🌟. En resumen, los rankings de palabras y
emojis dependen mucho de los anuncios, y los stickers casi no se usan para conversar.

## 6. Pregunta 4: ¿Qué días y a qué horas se envían más mensajes?

Telegram guardó las horas en UTC-07:00, la zona horaria de la computadora con la que
exporté el chat y
no la de los miembros. Por eso en la sección 2 las pasé a hora de Moscú (UTC+3, sin horario
de verano), y con esa hora saco tanto el día de la semana como la hora del día. El cambio
también mueve el día: un mensaje enviado a las 20:00 en UTC-07:00 cae a las 6:00 del día
siguiente en Moscú.

Esto es un supuesto. El grupo está en ruso, pero puede tener miembros en otras zonas
horarias, como Siberia, Kazajistán o Ucrania, así que los resultados describen el ritmo del
grupo visto desde Moscú y no la hora local de cada persona. Aquí cuento todos los mensajes,
incluidos los de "Cuenta eliminada" y los anuncios repetidos, porque la pregunta es sobre la
actividad del grupo. Como el periodo no tiene exactamente el mismo número de lunes, martes,
etc., comparo el promedio de mensajes por día y no el total.

In [ ]:
DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
dia = df["timestamp"].dt.dayofweek  # 0 = lunes
hora = df["timestamp"].dt.hour

# Todos los días del periodo, tengan o no mensajes
calendario = pd.date_range(df["timestamp"].min().normalize(),
                           df["timestamp"].max().normalize(), freq="D")
dias_en_periodo = pd.Series(calendario.dayofweek).value_counts()

tabla_p4 = pd.DataFrame({
    "mensajes": dia.value_counts(),
    "días en el periodo": dias_en_periodo,
    "promedio por día": dia.value_counts() / dias_en_periodo,
    "promedio sin repetidos": dia[~df["es_repetido"]].value_counts() / dias_en_periodo,
}).sort_index().set_axis(DIAS)

print(f"Periodo: del {calendario[0]:%d/%m/%Y} al {calendario[-1]:%d/%m/%Y} ({len(calendario)} días)")
tabla_p4.round(1)

In [ ]:
promedio_dia = len(df) / len(calendario)
promedio_hora = hora.value_counts().sort_index() / len(calendario)

print(f"Hora con más mensajes: {promedio_hora.idxmax()}:00 "
      f"({promedio_hora.max():.1f} mensajes en promedio)")
print(f"Hora con menos mensajes: {promedio_hora.idxmin()}:00 "
      f"({promedio_hora.min():.1f} mensajes en promedio)")
print(f"Mensajes enviados entre las 10:00 y las 23:59: {100 * hora.between(10, 23).mean():.1f} %")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [2, 3]})

ax1.bar(DIAS, tabla_p4["promedio por día"], color=AZUL)
ax1.bar_label(ax1.containers[0], fmt="{:.0f}", padding=3, fontsize=9,
              bbox=dict(facecolor="white", edgecolor="none", pad=1))
ax1.axhline(promedio_dia, color=GRIS, linestyle="--", zorder=0,
            label=f"Promedio general ({promedio_dia:.0f})")
ax1.set_ylim(0, tabla_p4["promedio por día"].max() * 1.2)
ax1.set_title("Mensajes promedio por día de la semana")
ax1.set_ylabel("Mensajes por día")
ax1.legend(loc="upper right")
ax1.grid(axis="x", visible=False)
plt.setp(ax1.get_xticklabels(), rotation=30, ha="right")

ax2.bar(promedio_hora.index, promedio_hora.values, color=AZUL)
ax2.set_title("Mensajes promedio por hora del día")
ax2.set_xlabel("Hora (Moscú)")
ax2.set_ylabel("Mensajes por hora")
ax2.set_xticks(range(0, 24, 2))
ax2.grid(axis="x", visible=False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "p4_dias_y_horas.png", dpi=120, bbox_inches="tight")
plt.show()

El heatmap cruza el día de la semana con la hora del día. Cada celda es el total de
mensajes que se enviaron en ese día y esa hora durante todo el periodo.

In [ ]:
conteo = pd.crosstab(dia, hora).set_axis(DIAS, axis=0)

fig, ax = plt.subplots(figsize=(13, 4.5))
sns.heatmap(conteo, cmap="Blues", linewidths=0.5, linecolor="white", ax=ax,
            cbar_kws={"label": "Mensajes"})
ax.set_title("Mensajes por día de la semana y hora del día")
ax.set_xlabel("Hora (Moscú)")
ax.set_ylabel("")
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "p4_heatmap_dia_hora.png", dpi=120, bbox_inches="tight")
plt.show()

### Interpretación

Por día de la semana no hay un patrón claro: el jueves es el día con más mensajes (92 en
promedio) y el miércoles el de menos (85), una diferencia de apenas 8 %, y el fin de semana
tiene casi el mismo promedio que los días entre semana, con o sin anuncios repetidos. Por
hora, en cambio, el ritmo es muy marcado: la actividad casi desaparece entre las 2:00 y las
6:00 de Moscú, sube desde las 9:00 y se mantiene alta de las 10:00 a las 23:00, con un pico a
las 15:00; en ese horario se envía el 80 % de los mensajes. El heatmap muestra que todos los
días siguen ese mismo ritmo, y las celdas más oscuras son el domingo y el jueves a las 15:00.
La actividad que sigue a medianoche puede venir de jugadores que se desvelan o de miembros
al este de Moscú, lo que recuerda que usar esa zona horaria es un supuesto.

## 7. Pregunta 5: ¿Qué palabras son las más usadas, en general y por usuario?

Antes de contar, lematizo los textos con spaCy (`ru_core_news_sm`), es decir, llevo cada palabra a su forma de diccionario. En ruso los sustantivos, adjetivos y verbos cambian mucho según el caso, el género o la persona; sin lematizar, "аккаунт", "аккаунта" y "аккаунте" (cuenta) o "ищу" e "искать" (buscar) contarían como palabras distintas y se repartirían el conteo. Uso los lemas en minúsculas, con la "ё" escrita como "е", sin la marca `@usuario_anon` y sin los anuncios repetidos, porque si no unos pocos textos pegados cientos de veces llenarían el top.

El modelo no lematiza bien la jerga del juego, así que 54 formas se corrigen con un
diccionario de equivalencias (por ejemplo, "акке" → "акк"). Las stop words son palabras muy comunes que casi no aportan significado, como "и" (y) o "на" (en): uso la lista de nltk para ruso más una lista propia de muletillas, pronombres y saludos escrita como lemas, y además quito las palabras de una sola letra. Las traducciones al español están en un archivo fijo, `references/traducciones_ru_es.csv`, en lugar de usar un traductor automático, que traduciría mal la jerga del juego.

In [ ]:
nltk.download("stopwords", quiet=True)

# Muletillas, pronombres y saludos del grupo que no están en la lista de nltk, como lemas
MULETILLAS = {
    "это", "мочь", "весь", "твой", "свой", "ваш", "который", "очень", "просто", "также",
    "вообще", "хотя", "либо", "пока", "сразу", "почему", "сколько", "что-то",
    "че", "щас", "хз", "мб", "пж", "плз", "пожалуйста", "привет", "вроде",
    "типа", "короче", "ладно", "кстати",
    "тебе", "всем",  # formas que el modelo deja sin lematizar
}
STOP_WORDS = {p.replace("ё", "е") for p in stopwords.words("russian")} | MULETILLAS

TRADUCCIONES = pd.read_csv(PROJ_ROOT / "references" / "traducciones_ru_es.csv",
                           index_col="ruso")["español"]


def contar(listas, quitar_stop=False):
    conteo = Counter()
    for lista in listas:
        for palabra in lista:
            if quitar_stop and (palabra in STOP_WORDS or len(palabra) == 1):
                continue
            conteo[palabra] += 1
    return pd.Series(conteo, dtype=int).sort_values(ascending=False)


def traducir(palabras):
    return [TRADUCCIONES.get(p, "?") for p in palabras]


def tabla_palabras(conteo, n=15):
    top = conteo.head(n)
    return pd.DataFrame({"lema": top.index, "traducción": traducir(top.index),
                         "veces": top.values}, index=range(1, len(top) + 1))


def etiquetar(conteo):
    # Para las gráficas: "palabra — traducción"
    return conteo.set_axis([f"{p} — {t}" for p, t in zip(conteo.index, traducir(conteo.index))])


print(f"Stop words: {len(STOP_WORDS)} ({len(MULETILLAS)} propias)")

In [ ]:
nlp = spacy.load("ru_core_news_sm", disable=["parser", "ner"])  # solo hacen falta la categoría y el lema

# El modelo no conoce la jerga del juego y deja muchas de sus formas sin lematizar;
# este diccionario de equivalencias las lleva a su forma de diccionario. También
# incluye verbos comunes que el modelo confunde con sustantivos y un error de dedo
# frecuente ("ищю").
CORRECCIONES = {
    # cuenta ("акк" es la abreviatura de "аккаунт")
    "акка": "акк", "акке": "акк", "акки": "акк", "акков": "акк", "акком": "акк",
    "аккаунты": "аккаунт", "аккаунта": "аккаунт", "аккаунте": "аккаунт",
    # skins limitados, legendarios y skins en general
    "лимиткой": "лимитка", "лимитку": "лимитка", "лимитки": "лимитка",
    "лимитками": "лимитка", "лимиток": "лимитка",
    "легой": "лега", "легу": "лега", "леги": "лега", "легами": "лега",
    "скины": "скин", "скинами": "скин", "скинов": "скин", "скином": "скин",
    # farmeo, cuentas vacías y otros términos del juego
    "фармом": "фарм", "фарма": "фарм", "фармлю": "фармить",
    "пустышку": "пустышка", "коллажи": "коллаж", "коллажей": "коллаж",
    "дайсов": "дайс", "дайсы": "дайс", "фрагов": "фраг", "фраги": "фраг",
    "приоры": "приор", "приоров": "приор", "приорами": "приор",
    "идене": "иден", "идена": "иден", "идену": "иден", "иденом": "иден",
    "отзывы": "отзыв", "чеки": "чек", "чека": "чек", "обмена": "обмен", "обмене": "обмен",
    "фулла": "фулл",
    # verbos y otras palabras frecuentes
    "ищу": "искать", "ищю": "искать", "куплю": "купить", "продам": "продать",
    "обменяю": "обменять", "обмени": "обменять", "сдам": "сдать", "пишите": "писать",
    "посмотрю": "посмотреть", "сексуальные": "сексуальный",
}


def lematizar(doc):
    lemas, adjetivos = [], []
    for token in doc:
        lema = token.lemma_.lower().replace("ё", "е")
        lema = CORRECCIONES.get(lema, lema)
        # Solo tokens que son palabras: sin números, emojis ni signos
        if not re.fullmatch(PATRON_PALABRA, lema):
            continue
        lemas.append(lema)
        if token.pos_ == "ADJ":
            adjetivos.append(lema)
    return lemas, adjetivos


# Cada texto distinto se procesa una sola vez (también los repetidos, para la comparación)
# y el resultado se reutiliza en la sección 8
unicos = df["texto"].dropna().unique()
sin_marca = pd.Series(unicos).str.replace("@usuario_anon", " ", regex=False)
lemas_de, adjetivos_de = {}, {}
for texto, doc in zip(unicos, nlp.pipe(sin_marca, batch_size=500)):
    lemas_de[texto], adjetivos_de[texto] = lematizar(doc)

textos = df[df["texto"].notna() & ~df["es_repetido"]].copy()
textos["lemas"] = textos["texto"].map(lemas_de)
textos["adjetivos"] = textos["texto"].map(adjetivos_de)
print(f"{len(unicos):,} textos distintos lematizados; {len(textos):,} mensajes sin repetidos")

### Palabras del grupo

Primero cuento los lemas de todo el grupo con stop words, y después sin ellas.

In [ ]:
con_stop = contar(textos["lemas"])
sin_stop = contar(textos["lemas"], quitar_stop=True)

print(f"{con_stop.sum():,} palabras en {len(textos):,} mensajes ({con_stop.size:,} lemas distintos)")
print(f"Stop words: {100 * (1 - sin_stop.sum() / con_stop.sum()):.1f} % de todas las palabras")

tabla_p5 = pd.concat({
    "Con stop words": tabla_palabras(con_stop),
    "Sin stop words": tabla_palabras(sin_stop),
}, axis=1)
tabla_p5

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
barras(ax1, etiquetar(con_stop.head(15)), "Con stop words", "Veces")
barras(ax2, etiquetar(sin_stop.head(15)), "Sin stop words", "Veces")
fig.suptitle("Las 15 palabras (lemas) más usadas en el grupo", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "p5_palabras_generales.png", dpi=120, bbox_inches="tight")
plt.show()

Como comparación rápida, así cambiaría el top 10 sin stop words si no quitara los anuncios
repetidos.

In [ ]:
todos = df["texto"].dropna().map(lemas_de)
con_repetidos = contar(todos, quitar_stop=True)

pd.DataFrame({
    "sin repetidos": etiquetar(sin_stop.head(10)).index,
    "con repetidos": etiquetar(con_repetidos.head(10)).index,
}, index=range(1, 11))

### Palabras por usuario

Las tablas muestran los 5 lemas más usados por cada uno de los 15 usuarios más activos y por
el resto agrupado como "Otros", con y sin stop words. Hulk aparece vacío porque todos sus
mensajes son anuncios repetidos.

In [ ]:
def top5(conteo):
    # Los 5 lemas más usados y su traducción, separados por comas
    top = conteo.head(5).index
    return {"lemas": ", ".join(top), "traducción": ", ".join(traducir(top))}


textos_usuarios = textos[textos["usuario"] != "Cuenta eliminada"]
grupo_texto = textos_usuarios["usuario"].where(textos_usuarios["usuario"].isin(top15), "Otros")

con_stop_usuario, sin_stop_usuario = {}, {}
for nombre in orden:
    listas = textos_usuarios.loc[grupo_texto == nombre, "lemas"]
    con_stop_usuario[nombre] = top5(contar(listas))
    sin_stop_usuario[nombre] = top5(contar(listas, quitar_stop=True))

pd.set_option("display.max_colwidth", None)
print("Con stop words")
pd.DataFrame(con_stop_usuario).T

In [ ]:
print("Sin stop words")
pd.DataFrame(sin_stop_usuario).T

### Interpretación

Con stop words, el top lo ocupan preposiciones y conjunciones como "на" (en), "с" (con) e
"и" (y), que son el 37.5 % de todas las palabras, aunque "акк" (cuenta), "искать" (buscar) y
"обмен" (intercambio) ya aparecen entre las 8 primeras. Sin stop words queda el vocabulario
de un mercado de cuentas: encabezan la lista "акк" e "искать", que al lematizar juntan
formas como "акка", "акке" o "ищу"; "обмен" (2,285) casi duplica a "продажа" (venta, 1,173),
y aparecen términos del juego como "скин" (skin), "лимитка" (skin limitado) y "пустышка"
(cuenta vacía). Si se incluyeran los anuncios repetidos, "фарм" (farmeo) subiría del lugar
14 al 5 y "отзыв" (reseña) del 15 al 6, porque vienen de anuncios de servicios pegados una y
otra vez. Por usuario, con stop words casi todos se parecen, pero sin ellas se ve a qué se
dedica cada uno: Batman, Superman y Aquaman anuncian collages, Flash y Rogue ofrecen farmeo,
Storm vende e intercambia, y Thor y Groot buscan o compran cuentas.

## 8. Pregunta 6: ¿Cuáles son los adjetivos más usados?

Reutilizo el procesamiento de spaCy de la sección 7: además del lema, el modelo etiqueta cada
palabra con su categoría gramatical, y aquí me quedo con las etiquetadas como `ADJ`. Contar
lemas es especialmente útil con los adjetivos, porque en ruso cambian según el género, el
número y el caso ("новый", "новая", "новые" cuentan como "новый"). Son los mismos mensajes,
sin repetidos ni menciones. El modelo pequeño se equivoca con la jerga, así que algunos
sustantivos que marca como adjetivos (como "акк") y los pronombres como "сам" (mismo) u
"другой" (otro) se quitan con una lista de exclusiones.

In [ ]:
# Sustantivos de jerga que el modelo confunde con adjetivos, y pronombres como "сам" o "другой"
NO_ADJETIVOS = {
    "акк", "лимитка", "приор", "мим", "скин",
    "сам", "сама", "сами", "само", "самый", "другой", "другая", "другие", "других",
    "остальной", "остальное", "остальные", "весь", "такой", "свой", "который",
    "каждый", "любой", "какой", "этот", "тот",
}

# Adjetivos de la sección 7; solo lemas en letras cirílicas
adjetivos = [a for lista in textos["adjetivos"] for a in lista
             if re.fullmatch(r"[а-я]+(?:-[а-я]+)*", a) and a not in NO_ADJETIVOS]

adjetivos_conteo = pd.Series(Counter(adjetivos)).sort_values(ascending=False)
print(f"{len(adjetivos):,} adjetivos ({adjetivos_conteo.size:,} lemas distintos)")
tabla_palabras(adjetivos_conteo, n=20).rename(columns={"lema": "adjetivo (lema)"})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
barras(ax, etiquetar(adjetivos_conteo.head(15)), "Los 15 adjetivos más usados", "Veces")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "p6_adjetivos.png", dpi=120, bbox_inches="tight")
plt.show()

### Interpretación

Los adjetivos más usados son "новый" (nuevo, 309 veces), "первый" (primero) y "второй" (segundo), seguidos de "желательный" (de "желательно", preferentemente) y "нужный" (necesario), que sirven para poner condiciones en los anuncios. Otros describen la cuenta o sus skins, como "небольшой" (pequeño), "жирный" (gorda, con mucho contenido) y "женский" (femenino, probablemente para describir skins). También hay adjetivos de opinión, como "хороший" (bueno), "крутой" (genial), "милый" (lindo) o "тупой" (tonto), pero la mayoría de los primeros lugares sirven para describir lo que se vende o se busca. El ranking hay que tomarlo con cuidado, porque el modelo pequeño de spaCy no conoce la jerga del juego y algunos errores se corrigen con una lista de exclusiones.

## 9. Preguntas adicionales

Además de las preguntas de la tarea, respondo tres preguntas propias sobre cómo funciona el
grupo: cómo cambió su actividad con el tiempo, si se usa más para comprar, vender o
intercambiar, y quién responde a quién. En las comparaciones por mes uso solo los meses
completos, de septiembre de 2025 a agosto de 2026, porque la exportación empieza el 30 de
agosto de 2025 y termina el 26 de septiembre de 2026.

### 9.1 ¿Cómo cambió la actividad del grupo con el tiempo?

Cuento por mes los mensajes, los usuarios activos (sin "Cuenta eliminada") y los usuarios
nuevos, que son los que aparecen por primera vez ese mes; en el primer mes, los "nuevos"
incluyen a miembros antiguos, porque ahí empieza la exportación. Como Telegram numera los
mensajes en orden, la diferencia entre el id más alto y el más bajo de cada mes estima todo
lo que se envió, incluidos los mensajes que después se borraron y los avisos del sistema. La
columna "% conservado" compara los mensajes que quedan con esa estimación.

In [ ]:
mes = df["timestamp"].dt.tz_localize(None).dt.to_period("M")
MESES = pd.period_range("2025-09", "2026-08", freq="M")  # solo meses completos

personas = df["usuario"] != "Cuenta eliminada"
primer_mes = mes[personas].groupby(df.loc[personas, "usuario"]).min()

mensual = pd.DataFrame({
    "mensajes": mes.value_counts(),
    # Telegram numera los mensajes en orden: del id más bajo al más alto del mes
    "ids asignados": df.groupby(mes)["id_mensaje"].agg(lambda ids: ids.max() - ids.min() + 1),
    "usuarios activos": df[personas].groupby(mes[personas])["usuario"].nunique(),
    "usuarios nuevos": primer_mes.value_counts(),
}).loc[MESES]
mensual["% conservado"] = (100 * mensual["mensajes"] / mensual["ids asignados"]).round(1)
mensual

In [ ]:
MESES_ES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
etiquetas_mes = [f"{MESES_ES[p.month - 1]} {p.year % 100}" for p in MESES]

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
axes[0].plot(etiquetas_mes, mensual["ids asignados"], marker="o", color=GRIS, linestyle="--",
             label="Ids asignados (incluye borrados y avisos)")
axes[0].plot(etiquetas_mes, mensual["mensajes"], marker="o", color=AZUL, linewidth=2,
             label="Mensajes en la exportación")
axes[0].set_title("Mensajes por mes")
axes[0].legend(loc="lower right")

axes[1].plot(etiquetas_mes, mensual["usuarios activos"], marker="o", color=AZUL, linewidth=2)
axes[1].set_title("Usuarios activos por mes")

axes[2].bar(etiquetas_mes, mensual["usuarios nuevos"], color=AZUL)
axes[2].set_title("Usuarios nuevos por mes (primer mensaje en la exportación)")
axes[2].set_xlabel("Mes (hora de Moscú)")

for ax in axes:
    ax.set_ylim(bottom=0)
    ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "p7_actividad_mensual.png", dpi=120, bbox_inches="tight")
plt.show()

#### Interpretación

A simple vista el grupo crece: los mensajes que quedan pasan de 2,081 en septiembre de 2025
a unos 3,000 al mes en 2026. Pero los ids asignados, que incluyen lo que después se borró, no
crecen igual: suben hasta 5,634 en enero de 2026 y bajan a 3,833 en agosto, mientras que el
porcentaje conservado pasa de entre 50 % y 57 % en 2025 a 78 % en agosto de 2026, así que
buena parte del crecimiento aparente se explica porque de los meses antiguos se han borrado
más mensajes. Los usuarios activos se mantienen entre 184 y 251 al mes hasta julio, y agosto
de 2026 destaca con 319 activos y 139 nuevos, el máximo después del primer mes, aunque
puede estar inflado por el mismo efecto, porque ese mes conserva más mensajes. Los 210
"nuevos" de septiembre de 2025 también están inflados, porque ahí empieza la
exportación.

### 9.2 ¿Es un grupo más de compra, de venta o de intercambio?

Clasifico los mensajes con texto (sin repetidos) según sus lemas de la sección 7. Un mensaje
es de venta si tiene "продать", "продажа" o "продавать"; de intercambio si tiene "обмен",
"обменять" u "обменивать"; de compra si tiene "купить" o "покупка"; y de servicios si
ofrece farmeo o collages ("фарм", "фармить", "коллаж"). "Искать" (buscar) también cuenta
como compra, pero solo si el mensaje no tiene palabras de venta ni de intercambio, porque en
un anuncio de intercambio lo que se busca es algo a cambio. Un mensaje puede tener varias
categorías, así que los porcentajes no suman 100.

In [ ]:
PALABRAS_CLAVE = {
    "compra": {"купить", "покупка"},
    "venta": {"продать", "продажа", "продавать"},
    "intercambio": {"обмен", "обменять", "обменивать"},
    "servicios": {"фарм", "фармить", "коллаж"},
}
# Paleta Okabe-Ito, pensada para personas con daltonismo
COLORES_CATEGORIA = {"compra": "#0072B2", "venta": "#D55E00",
                     "intercambio": "#009E73", "servicios": "#CC79A7"}


def categorias(lemas):
    lemas = set(lemas)
    encontradas = {nombre for nombre, claves in PALABRAS_CLAVE.items() if lemas & claves}
    # "искать" (buscar) cuenta como compra solo si no se habla de vender ni de intercambiar
    if "искать" in lemas and not encontradas & {"venta", "intercambio"}:
        encontradas.add("compra")
    return encontradas


cats = textos["lemas"].map(categorias)
clasificacion = pd.DataFrame({nombre: cats.map(lambda c, n=nombre: n in c) for nombre in PALABRAS_CLAVE})
clasificacion["sin categoría"] = cats.map(len) == 0

print(f"{len(textos):,} mensajes con texto (sin repetidos)")
pd.DataFrame({
    "mensajes": clasificacion.sum(),
    "% de los mensajes": (100 * clasificacion.mean()).round(1),
})

In [ ]:
por_mes = 100 * clasificacion.groupby(mes.loc[textos.index]).mean()
por_mes = por_mes.loc[MESES, list(PALABRAS_CLAVE)]

fig, ax = plt.subplots(figsize=(12, 5))
for categoria, color in COLORES_CATEGORIA.items():
    ax.plot(etiquetas_mes, por_mes[categoria], marker="o", linewidth=2, color=color, label=categoria)
    # Etiqueta directa al final de cada línea
    ax.annotate(categoria, (len(MESES) - 1, por_mes[categoria].iloc[-1]), xytext=(8, 0),
                textcoords="offset points", va="center", color=color, fontsize=10)
ax.set_ylim(0, por_mes.max().max() * 1.15)
ax.set_xlim(-0.5, len(MESES) + 0.8)
ax.set_title("Mensajes de compra, venta, intercambio y servicios por mes")
ax.set_ylabel("% de los mensajes con texto")
ax.legend(loc="lower left", ncol=4)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "p7_compra_venta_intercambio.png", dpi=120, bbox_inches="tight")
plt.show()

#### Interpretación

Con este criterio, el grupo es más de compra que de venta: el 24.9 % de los mensajes con
texto busca o compra una cuenta, el 20.5 % habla de intercambio y solo el 16.6 % de venta,
mientras que el 9.1 % ofrece servicios como farmeo o collages; el 40 % restante no cae en
ninguna categoría. Por mes, la compra tiene picos en enero (35.2 %) y marzo de 2026 (41.7 %),
el intercambio llega a su máximo en febrero de 2026 (30.2 %) y después baja hasta 12.7 % en
agosto, igual que los servicios, que caen a 2.8 %. Como la clasificación depende de palabras
clave, estos porcentajes son una estimación: hay anuncios que no usan ninguna de estas
palabras y otros que mencionan varias cosas a la vez. Además, los porcentajes se calculan
sobre los mensajes que siguen en el grupo, así que si algún tipo de anuncio se borra más
que otro, la tendencia podría cambiar.

### 9.3 ¿Quién responde más y quién recibe más respuestas?

Una respuesta guarda el id del mensaje al que responde, pero no siempre ese mensaje está en
la exportación. Telegram numera los mensajes de un grupo en orden, así que el id permite
saber a qué apunta cada respuesta: a un mensaje exportado, a un aviso del sistema, a un
mensaje anterior al 30 de agosto de 2025 (si el id es menor que el primero exportado) o a un
id dentro del rango que ya no existe en el HTML.

In [ ]:
# Ids de los avisos del sistema (p. ej. alguien se unió); los separadores de fecha tienen ids negativos
ids_avisos = set()
for ruta in archivos:
    for aviso in html.parse(str(ruta)).getroot().find_class("service"):
        if aviso.get("id", "").startswith("message"):
            numero = int(aviso.get("id").replace("message", ""))
            if numero > 0:
                ids_avisos.add(numero)

ids_usuario = set(df["id_mensaje"])
primer_id, ultimo_id = df["id_mensaje"].min(), df["id_mensaje"].max()


def destino(id_respondido):
    if id_respondido in ids_usuario:
        return "mensaje de la exportación"
    if id_respondido in ids_avisos:
        return "aviso del sistema"
    if id_respondido < primer_id:
        return "anterior a la exportación"
    return "no existe en la exportación"


respuestas = df[df["es_respuesta"]].copy()
respuestas["destino"] = respuestas["responde_a_id"].map(destino)

# Ids del rango exportado que no son ni mensajes de usuario ni avisos del sistema
ids_rango = ultimo_id - primer_id + 1
existentes = len(ids_usuario) + len([i for i in ids_avisos if primer_id <= i <= ultimo_id])
print(f"{len(respuestas):,} respuestas ({100 * len(respuestas) / len(df):.1f} % de los mensajes)")
print(f"Ids del {primer_id} al {ultimo_id}: {ids_rango:,}; sin mensaje ni aviso: "
      f"{ids_rango - existentes:,} ({100 * (ids_rango - existentes) / ids_rango:.1f} %)")

destinos = ["mensaje de la exportación", "no existe en la exportación",
            "anterior a la exportación", "aviso del sistema"]
conteo = respuestas["destino"].value_counts().reindex(destinos, fill_value=0)
pd.DataFrame({"respuestas": conteo, "%": (100 * conteo / len(respuestas)).round(1)})

No se puede saber por qué se borraron esos mensajes, pero en un grupo de compraventa es
probable que sean anuncios de cuentas ya vendidas, anuncios que se borran para volver a
publicarlos o mensajes eliminados por moderadores o bots. Por eso las respuestas enviadas
se cuentan con todas, pero las recibidas y el tiempo de respuesta solo con las que apuntan a
mensajes de la exportación, porque en las demás no se conoce el mensaje original. En las
recibidas tampoco cuento las respuestas a uno mismo, que probablemente sirven para
volver a subir un anuncio propio.

In [ ]:
enviadas = respuestas.loc[respuestas["usuario"] != "Cuenta eliminada", "usuario"].value_counts()

# Recibidas: solo respuestas a mensajes que están en la exportación
en_exportacion = respuestas[respuestas["destino"] == "mensaje de la exportación"].copy()
autor_de = df.set_index("id_mensaje")["usuario"]
en_exportacion["autor_original"] = en_exportacion["responde_a_id"].map(autor_de)
a_si_mismo = en_exportacion["autor_original"] == en_exportacion["usuario"]
a_otros = en_exportacion[~a_si_mismo]
recibidas = a_otros.loc[a_otros["autor_original"] != "Cuenta eliminada", "autor_original"].value_counts()

print(f"Respuestas enviadas: {len(respuestas):,}; a mensajes de la exportación: "
      f"{len(en_exportacion):,}; a sí mismo: {a_si_mismo.sum()} ({100 * a_si_mismo.mean():.1f} %)")
print(f"Usuarios que respondieron al menos una vez: {len(enviadas)}; "
      f"que recibieron al menos una respuesta: {len(recibidas)}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
barras(ax1, enviadas.head(10), "Respuestas enviadas", "Respuestas")
barras(ax2, recibidas.head(10), "Respuestas recibidas de otros usuarios", "Respuestas")
fig.suptitle("Los 10 usuarios que más responden y más respuestas reciben", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "p7_respuestas.png", dpi=120, bbox_inches="tight")
plt.show()

Por último, mido cuánto tarda una respuesta: el tiempo entre el mensaje original y la
respuesta, con las mismas respuestas a otros usuarios. Uso escala logarítmica porque los
tiempos van de segundos a meses; las respuestas enviadas en el mismo segundo no caben en esa
escala y quedan fuera del boxplot.

In [ ]:
fecha_de = df.set_index("id_mensaje")["timestamp"]
minutos = (a_otros["timestamp"] - a_otros["responde_a_id"].map(fecha_de)).dt.total_seconds() / 60

print(f"{len(minutos):,} respuestas a otros usuarios; {(minutos == 0).sum()} en el mismo segundo")
print(f"Mediana: {minutos.median():.0f} min; la mitad central, entre "
      f"{minutos.quantile(0.25):.0f} y {minutos.quantile(0.75):.0f} min")
print(f"En menos de 1 hora: {100 * (minutos < 60).mean():.1f} %; "
      f"después de 1 día: {100 * (minutos > 60 * 24).mean():.1f} %")

fig, ax = plt.subplots(figsize=(11, 3))
sns.boxplot(x=minutos[minutos > 0], ax=ax, color=AZUL, log_scale=True, width=0.4)
marcas = {"1 min": 1, "10 min": 10, "1 h": 60, "1 día": 60 * 24, "1 semana": 60 * 24 * 7,
          "1 mes": 60 * 24 * 30}
ax.set_xticks(list(marcas.values()), list(marcas.keys()))
ax.set_title("Tiempo entre un mensaje y su respuesta")
ax.set_xlabel("Tiempo de respuesta (escala logarítmica)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "p7_tiempo_respuesta.png", dpi=120, bbox_inches="tight")
plt.show()

#### Interpretación

Solo el 12.9 % de los mensajes son respuestas, y el 44.9 % de ellas apunta a mensajes que ya
no existen; ninguna apunta a mensajes anteriores a la exportación ni a avisos del sistema, y
en el rango exportado falta el 39.9 % de los ids, así que lo más probable es que esos
mensajes se hayan borrado. Storm (137) y Thor (121), dos de los 15 más activos, son quienes
más responden y también están entre quienes más respuestas reciben, mientras que Hawkeye,
que está justo fuera de los 15 más activos pero es quien más palabras escribe sin
repetidos, es quien más respuestas recibe (65). Las respuestas son rápidas: la mediana es de
16 minutos, la mitad central está entre 4 y 63 minutos y el 73.9 % llega en menos de una
hora, lo que cuadra con un mercado donde una oferta pierde interés pronto.